# Bsr Sales Estimator Lumaiscope

Converts each product's Best Sellers Rank into estimated monthly units and revenue using LumaiScope's public BSR-estimate endpoint, returning a central estimate with lower and upper bounds.

**Setup.** Data paths point to `/content/drive/MyDrive/YOUR_PROJECT_FOLDER/...` — change `YOUR_PROJECT_FOLDER` to your own folder before running. API keys are read from Colab Secrets / environment variables, never hard-coded.

> Code comments and console messages are in Korean (original working language). See the [folder README](README.md) for the pipeline overview and the [disclaimer](../README.md#disclaimer).


> ⚠️ **Read before running.** This notebook sends each product's rank and price to LumaiScope's public BSR-estimate endpoint. Check LumaiScope's terms and rate limits before using it.
>
> It is shared to show the collection and parsing method, not as a tool to run against api.lumaiscope.com. Do not run it against the live site without that site's permission. To follow the downstream steps, use the synthetic files in [`sample-data/`](../sample-data/).

In [ ]:
# ============================================================
# LumaiScope BSR Sales Estimator 연결 버전
#
# 입력:
#   Amazon MERGE Excel 1개
#
# 사용 데이터:
#   ASIN
#   BSR_대분류
#   BSR_대분류명
#   BSR_하위순위
#   BSR_하위분류명
#   월간구매표시_원문
#   월간구매표시_하한
#   판매가격
#
# API:
#   https://api.lumaiscope.com/api/public/bsr-estimate
#
# 결과:
#   BSR기반_월간판매량_추정
#   BSR기반_월간판매량_하한
#   BSR기반_월간판매량_상한
#   BSR기반_월간매출_USD
#   BSR추정_일판매량
#   BSR추정_모델
#   BSR추정_표본수
#   BSR추정_R2
#   BSR추정_기준일
#   BSR추정_외삽여부
#
# ============================================================


# ============================================================
# 0. 라이브러리
# ============================================================

import pandas as pd
import numpy as np
import requests
import re
import time
import json
import os

from datetime import datetime
from google.colab import files


# ============================================================
# 1. Excel 업로드
# ============================================================

print("=" * 80)
print("📂 원본 Amazon MERGE Excel 파일 1개를 업로드하세요.")
print("=" * 80)

uploaded = files.upload()

excel_files = [
    f for f in uploaded.keys()
    if f.lower().endswith((".xlsx", ".xls"))
]

if not excel_files:
    raise ValueError("Excel 파일이 없습니다.")

input_file = excel_files[0]

print(f"\n✅ 입력 파일: {input_file}")


# ============================================================
# 2. Excel 시트 확인
# ============================================================

xls = pd.ExcelFile(input_file)

print("\n📑 시트 목록")

for s in xls.sheet_names:
    print(
        f" - {s}"
    )


# ============================================================
# 3. 분석 시트 자동 선택
#
# 원본 MERGE 파일이면 보통 Sheet1
# 이전 분석 파일이면 원본데이터가 있을 수 있음
# ============================================================

preferred_sheets = [
    "Sheet1",
    "원본데이터",
]

selected_sheet = None

for s in preferred_sheets:

    if s not in xls.sheet_names:
        continue

    temp = pd.read_excel(
        input_file,
        sheet_name=s,
        nrows=5
    )

    cols = set(
        str(c).strip()
        for c in temp.columns
    )

    # BSR 원본 컬럼이 있는 시트를 우선
    if (
        "BSR_대분류" in cols
        and
        "ASIN" in cols
    ):
        selected_sheet = s
        break


# 못 찾으면 가장 많은 행의 시트 선택
if selected_sheet is None:

    sheet_sizes = {}

    for s in xls.sheet_names:

        temp = pd.read_excel(
            input_file,
            sheet_name=s
        )

        sheet_sizes[s] = len(temp)

    selected_sheet = max(
        sheet_sizes,
        key=sheet_sizes.get
    )


df = pd.read_excel(
    input_file,
    sheet_name=selected_sheet
)

df.columns = [
    str(c).strip()
    for c in df.columns
]

print(
    f"\n✅ 분석 시트: {selected_sheet}"
)

print(
    f"✅ 상품 수: {len(df):,}"
)


# ============================================================
# 4. 컬럼 자동 선택 함수
# ============================================================

def choose_col(candidates):

    for c in candidates:

        if c in df.columns:
            return c

    return None


COL_ASIN = choose_col([
    "ASIN",
    "ASIN_분석"
])

COL_BSR = choose_col([
    "BSR_대분류",
    "BSR_대분류_분석"
])

COL_BSR_NAME = choose_col([
    "BSR_대분류명",
    "BSR_대분류명_분석"
])

COL_BSR_SUB = choose_col([
    "BSR_하위순위",
    "BSR_하위순위_분석"
])

COL_BSR_SUB_NAME = choose_col([
    "BSR_하위분류명",
    "BSR_하위분류명_분석"
])

COL_MONTHLY_TEXT = choose_col([
    "월간구매표시_원문",
    "월간구매표시_원문_분석"
])

COL_MONTHLY_VALUE = choose_col([
    "월간구매표시_하한",
    "월간구매표시_하한_분석"
])

COL_PRICE = choose_col([
    "판매가격",
    "판매가격_USD",
    "판매가격_USD_분석",
    "현재가격_USD"
])

COL_BRAND = choose_col([
    "브랜드_file1",
    "브랜드_file2",
    "브랜드",
])

COL_PRODUCT = choose_col([
    "상품명_file1",
    "상품명_file2",
    "상품명",
])


print("\n" + "=" * 80)
print("🔎 컬럼 매칭")
print("=" * 80)

print("ASIN:", COL_ASIN)
print("BSR:", COL_BSR)
print("BSR 카테고리:", COL_BSR_NAME)
print("BSR 하위순위:", COL_BSR_SUB)
print("월간구매표시:", COL_MONTHLY_TEXT)
print("월간구매표시 하한:", COL_MONTHLY_VALUE)
print("판매가격:", COL_PRICE)


# ============================================================
# 5. 필수 컬럼 검사
# ============================================================

required = {
    "ASIN": COL_ASIN,
    "BSR_대분류": COL_BSR,
    "BSR_대분류명": COL_BSR_NAME,
}

missing = [
    name
    for name, col in required.items()
    if col is None
]

if missing:

    raise ValueError(
        "필수 컬럼이 없습니다:\n"
        + "\n".join(missing)
        + "\n\n"
        "반드시 원본 MERGE Excel을 사용하세요."
    )


# ============================================================
# 6. 숫자 변환
# ============================================================

def to_number(value):

    if pd.isna(value):
        return np.nan

    s = str(value).strip()

    if not s:
        return np.nan

    s = (
        s.replace("$", "")
         .replace(",", "")
         .replace("₩", "")
    )

    m = re.search(
        r"-?\d+(?:\.\d+)?",
        s
    )

    if not m:
        return np.nan

    try:
        return float(
            m.group()
        )
    except:
        return np.nan


def parse_bsr(value):

    if pd.isna(value):
        return np.nan

    s = str(value).strip()

    # #13
    m = re.search(
        r"#\s*([\d,]+)",
        s
    )

    if m:

        return float(
            m.group(1).replace(",", "")
        )

    # 숫자만 있는 경우
    s = s.replace(",", "")

    m = re.search(
        r"\d+",
        s
    )

    if m:

        return float(
            m.group()
        )

    return np.nan


# ============================================================
# 7. 분석용 컬럼 생성
# ============================================================

df["ASIN_분석"] = (
    df[COL_ASIN]
    .astype(str)
    .str.strip()
    .str.upper()
)

df["BSR_분석"] = (
    df[COL_BSR]
    .apply(parse_bsr)
)

df["BSR_대분류명_분석"] = (
    df[COL_BSR_NAME]
    .fillna("")
    .astype(str)
    .str.strip()
)

if COL_BSR_SUB is not None:

    df["BSR_하위순위_분석"] = (
        df[COL_BSR_SUB]
        .apply(parse_bsr)
    )

else:

    df["BSR_하위순위_분석"] = np.nan


if COL_BSR_SUB_NAME is not None:

    df["BSR_하위분류명_분석"] = (
        df[COL_BSR_SUB_NAME]
        .fillna("")
        .astype(str)
        .str.strip()
    )

else:

    df["BSR_하위분류명_분석"] = ""


if COL_MONTHLY_TEXT is not None:

    df["월간구매표시_원문_분석"] = (
        df[COL_MONTHLY_TEXT]
        .fillna("")
        .astype(str)
        .str.strip()
    )

else:

    df["월간구매표시_원문_분석"] = ""


if COL_MONTHLY_VALUE is not None:

    df["월간구매표시_하한_분석"] = (
        df[COL_MONTHLY_VALUE]
        .apply(to_number)
    )

else:

    df["월간구매표시_하한_분석"] = np.nan


if COL_PRICE is not None:

    df["판매가격_USD_분석"] = (
        df[COL_PRICE]
        .apply(to_number)
    )

else:

    df["판매가격_USD_분석"] = np.nan


# ============================================================
# 8. ASIN 중복 제거
# ============================================================

before = len(df)

df = (
    df
    .drop_duplicates(
        subset=["ASIN_분석"],
        keep="first"
    )
    .copy()
)

print(
    f"\n🧹 ASIN 중복: "
    f"{before:,} → {len(df):,}"
)


# ============================================================
# 9. LumaiScope 카테고리 매핑
#
# LumaiScope API의 category 값
# ============================================================

def map_lumaiscope_category(
    bsr_name,
    market_category=""
):

    s = str(bsr_name).strip()

    # --------------------------------------------------------
    # Beauty
    # --------------------------------------------------------

    if s in [
        "Beauty & Personal Care",
        "Beauty",
    ]:

        return "Beauty"


    # --------------------------------------------------------
    # Health & Personal Care
    # --------------------------------------------------------

    if s in [
        "Health & Household",
        "Health & Personal Care",
    ]:

        return "Health & Personal Care"


    # --------------------------------------------------------
    # Home & Kitchen
    # --------------------------------------------------------

    if s in [
        "Home & Kitchen",
    ]:

        return "Home & Kitchen"


    # --------------------------------------------------------
    # Kitchen
    # --------------------------------------------------------

    if s in [
        "Kitchen & Dining",
        "Kitchen",
    ]:

        return "Kitchen"


    # --------------------------------------------------------
    # Sports
    # --------------------------------------------------------

    if s in [
        "Sports & Outdoors",
        "Sports",
    ]:

        return "Sports"


    # --------------------------------------------------------
    # Toys
    # --------------------------------------------------------

    if s in [
        "Toys & Games",
        "Toys",
    ]:

        return "Toys & Games"


    # --------------------------------------------------------
    # Electronics
    # --------------------------------------------------------

    if s in [
        "Electronics",
    ]:

        return "Electronics"


    # --------------------------------------------------------
    # Pet
    # --------------------------------------------------------

    if s in [
        "Pet Supplies",
    ]:

        return "Pet Supplies"


    # --------------------------------------------------------
    # Baby
    # --------------------------------------------------------

    if s in [
        "Baby",
        "Baby Products",
    ]:

        return "Baby Products"


    # --------------------------------------------------------
    # Garden
    # --------------------------------------------------------

    if s in [
        "Garden & Outdoor",
    ]:

        return "Garden & Outdoor"


    # --------------------------------------------------------
    # Clothing
    # --------------------------------------------------------

    if s in [
        "Clothing, Shoes & Jewelry",
        "Clothing & Accessories",
    ]:

        return "Clothing & Accessories"


    # --------------------------------------------------------
    # Office
    # --------------------------------------------------------

    if s in [
        "Office Products",
    ]:

        return "Office Products"


    # --------------------------------------------------------
    # Tools
    # --------------------------------------------------------

    if s in [
        "Tools & Home Improvement",
    ]:

        return "Tools & Home Improvement"


    # --------------------------------------------------------
    # Grocery
    # --------------------------------------------------------

    if s in [
        "Grocery & Gourmet Food",
        "Grocery & Gourmet",
    ]:

        return "Grocery & Gourmet"


    # --------------------------------------------------------
    # Automotive
    # --------------------------------------------------------

    if s in [
        "Automotive",
    ]:

        return "Automotive"


    # --------------------------------------------------------
    # Books
    # --------------------------------------------------------

    if s in [
        "Books",
    ]:

        return "Books"


    return None


df["LumaiScope_카테고리"] = df.apply(
    lambda row:
        map_lumaiscope_category(
            row["BSR_대분류명_분석"]
        ),
    axis=1
)


# ============================================================
# 10. 카테고리 매핑 결과 확인
# ============================================================

print("\n" + "=" * 80)
print("📊 LumaiScope 카테고리 매핑")
print("=" * 80)

print(
    df[
        [
            "BSR_대분류명_분석",
            "LumaiScope_카테고리"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "BSR_대분류명_분석"
    )
    .to_string(index=False)
)


# ============================================================
# 11. API 호출 함수
# ============================================================

API_URL = (
    "https://api.lumaiscope.com/"
    "api/public/bsr-estimate"
)


session = requests.Session()

session.headers.update({

    "User-Agent":
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/154.0.0.0 "
        "Safari/537.36",

    "Accept":
        "application/json",

})


def call_lumaiscope(
    bsr,
    category,
    marketplace="us"
):

    if pd.isna(bsr):
        return {
            "status": "NO_BSR"
        }

    if not category:
        return {
            "status": "NO_CATEGORY"
        }

    try:

        params = {

            "bsr": int(bsr),

            "marketplace":
                marketplace,

            "category":
                category

        }

        response = session.get(
            API_URL,
            params=params,
            timeout=30
        )

        if response.status_code != 200:

            return {

                "status":
                    f"HTTP_{response.status_code}",

                "http_status":
                    response.status_code,

                "response":
                    response.text[:500]

            }

        data = response.json()

        return {

            "status":
                "SUCCESS",

            "monthly":
                data.get("monthly"),

            "daily":
                data.get("daily"),

            "low":
                data.get("low"),

            "high":
                data.get("high"),

            "basis":
                data.get("basis"),

            "marketplace":
                data.get("marketplace"),

            "category":
                data.get("category"),

            "sample_size":
                data.get("sample_size"),

            "r_squared":
                data.get("r_squared"),

            "fitted_at":
                data.get("fitted_at"),

            "extrapolated":
                data.get("extrapolated"),

            "api_bsr":
                data.get("bsr"),

        }

    except Exception as e:

        return {

            "status":
                "ERROR",

            "error":
                str(e)

        }


# ============================================================
# 12. API 호출 전 상태
# ============================================================

api_candidates = df[
    df["BSR_분석"].notna()
    &
    df["LumaiScope_카테고리"].notna()
].copy()


print("\n" + "=" * 80)
print("🌐 LumaiScope API 호출 대상")
print("=" * 80)

print(
    f"전체 상품: {len(df):,}"
)

print(
    f"BSR 존재: "
    f"{df['BSR_분석'].notna().sum():,}"
)

print(
    f"BSR + API 카테고리 존재: "
    f"{len(api_candidates):,}"
)


# ============================================================
# 13. API 호출
#
# 같은 BSR + category가 여러 개면
# API를 매번 호출하지 않고 캐시 사용
# ============================================================

cache = {}

results = []

total = len(df)

print("\n" + "=" * 80)
print("🚀 LumaiScope API 호출 시작")
print("=" * 80)


for idx, row in df.iterrows():

    bsr = row["BSR_분석"]

    category = row[
        "LumaiScope_카테고리"
    ]

    cache_key = None

    if (
        pd.notna(bsr)
        and category
    ):

        cache_key = (
            int(bsr),
            category
        )


    # --------------------------------------------------------
    # 캐시
    # --------------------------------------------------------

    if cache_key in cache:

        result = cache[
            cache_key
        ].copy()

        result[
            "from_cache"
        ] = True

    else:

        result = call_lumaiscope(
            bsr=bsr,
            category=category,
            marketplace="us"
        )

        result[
            "from_cache"
        ] = False

        if cache_key is not None:

            cache[
                cache_key
            ] = result.copy()

        # API 과도 호출 방지
        time.sleep(0.10)


    results.append(result)


    # 진행상황
    current = len(results)

    if (
        current == 1
        or current % 10 == 0
        or current == total
    ):

        success_count = sum(
            r.get("status") == "SUCCESS"
            for r in results
        )

        print(
            f"[{current:,}/{total:,}] "
            f"SUCCESS={success_count:,} "
            f"CACHE={sum(r.get('from_cache', False) for r in results):,}"
        )


# ============================================================
# 14. API 결과 DataFrame
# ============================================================

api_df = pd.DataFrame(
    results,
    index=df.index
)


# ============================================================
# 15. API 결과 컬럼 생성
# ============================================================

df["BSR_API_상태"] = (
    api_df["status"]
)

df["BSR추정_월간판매량"] = (
    pd.to_numeric(
        api_df.get("monthly"),
        errors="coerce"
    )
)

df["BSR추정_일판매량"] = (
    pd.to_numeric(
        api_df.get("daily"),
        errors="coerce"
    )
)

df["BSR추정_하한"] = (
    pd.to_numeric(
        api_df.get("low"),
        errors="coerce"
    )
)

df["BSR추정_상한"] = (
    pd.to_numeric(
        api_df.get("high"),
        errors="coerce"
    )
)

df["BSR추정_모델"] = (
    api_df.get("basis")
)

df["BSR추정_마켓"] = (
    api_df.get("marketplace")
)

df["BSR추정_API카테고리"] = (
    api_df.get("category")
)

df["BSR추정_표본수"] = (
    pd.to_numeric(
        api_df.get("sample_size"),
        errors="coerce"
    )
)

df["BSR추정_R2"] = (
    pd.to_numeric(
        api_df.get("r_squared"),
        errors="coerce"
    )
)

df["BSR추정_기준일"] = (
    api_df.get("fitted_at")
)

df["BSR추정_외삽여부"] = (
    api_df.get("extrapolated")
)


# ============================================================
# 16. 최종 컬럼
# ============================================================

df[
    "BSR기반_월간판매량_추정"
] = df[
    "BSR추정_월간판매량"
]


df[
    "BSR기반_월간판매량_하한"
] = df[
    "BSR추정_하한"
]


df[
    "BSR기반_월간판매량_상한"
] = df[
    "BSR추정_상한"
]


# ============================================================
# 17. 월간 매출
#
# 월간판매량 추정 × 현재 판매가격
# ============================================================

df[
    "BSR기반_월간매출_USD"
] = (
    df[
        "BSR기반_월간판매량_추정"
    ]
    *
    df[
        "판매가격_USD_분석"
    ]
)


# ============================================================
# 18. 매출 하한 / 상한
# ============================================================

df[
    "BSR기반_월간매출_하한_USD"
] = (
    df[
        "BSR기반_월간판매량_하한"
    ]
    *
    df[
        "판매가격_USD_분석"
    ]
)


df[
    "BSR기반_월간매출_상한_USD"
] = (
    df[
        "BSR기반_월간판매량_상한"
    ]
    *
    df[
        "판매가격_USD_분석"
    ]
)


# ============================================================
# 19. 숫자 반올림
# ============================================================

for col in [
    "BSR기반_월간판매량_추정",
    "BSR기반_월간판매량_하한",
    "BSR기반_월간판매량_상한",
    "BSR추정_일판매량",
]:

    df[col] = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .round(0)
    )


for col in [
    "BSR기반_월간매출_USD",
    "BSR기반_월간매출_하한_USD",
    "BSR기반_월간매출_상한_USD",
]:

    df[col] = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .round(2)
    )


# ============================================================
# 20. 상태
# ============================================================

def make_status(row):

    if row["BSR_API_상태"] == "SUCCESS":

        if bool(row["BSR추정_외삽여부"]):

            return "LumaiScope API 성공 / 외삽"

        return "LumaiScope API 성공"

    if row["BSR_API_상태"] == "NO_BSR":

        return "BSR 없음"

    if row["BSR_API_상태"] == "NO_CATEGORY":

        return "LumaiScope 카테고리 미매핑"

    return (
        "API 오류: "
        + str(row["BSR_API_상태"])
    )


df["BSR모델_상태"] = df.apply(
    make_status,
    axis=1
)


# ============================================================
# 21. 결과 요약
# ============================================================

success_mask = (
    df["BSR_API_상태"]
    == "SUCCESS"
)

print("\n" + "=" * 80)
print("📊 API 결과")
print("=" * 80)

print(
    f"전체 상품: "
    f"{len(df):,}"
)

print(
    f"API 성공: "
    f"{success_mask.sum():,}"
)

print(
    f"API 실패/미실행: "
    f"{(~success_mask).sum():,}"
)

print(
    f"API 캐시 사용 횟수: "
    f"{sum(r.get('from_cache', False) for r in results):,}"
)


# ============================================================
# 22. 예상 매출 합계
# ============================================================

total_sales = (
    df[
        "BSR기반_월간판매량_추정"
    ]
    .sum(
        skipna=True
    )
)


total_revenue = (
    df[
        "BSR기반_월간매출_USD"
    ]
    .sum(
        skipna=True
    )
)


total_revenue_low = (
    df[
        "BSR기반_월간매출_하한_USD"
    ]
    .sum(
        skipna=True
    )
)


total_revenue_high = (
    df[
        "BSR기반_월간매출_상한_USD"
    ]
    .sum(
        skipna=True
    )
)


print(
    f"\nBSR 기반 추정 월간 판매량: "
    f"{total_sales:,.0f}"
)

print(
    f"BSR 기반 추정 월간 매출: "
    f"${total_revenue:,.2f}"
)

print(
    f"BSR 기반 추정 월간 매출 하한: "
    f"${total_revenue_low:,.2f}"
)

print(
    f"BSR 기반 추정 월간 매출 상한: "
    f"${total_revenue_high:,.2f}"
)


# ============================================================
# 23. 상품통합
# ============================================================

상품통합 = pd.DataFrame({

    "ASIN":
        df["ASIN_분석"].values,

    "브랜드":
        (
            df[COL_BRAND].values
            if COL_BRAND
            else ""
        ),

    "상품명":
        (
            df[COL_PRODUCT].values
            if COL_PRODUCT
            else ""
        ),

    "판매가격_USD":
        df["판매가격_USD_분석"].values,

    "월간구매표시_원문":
        df[
            "월간구매표시_원문_분석"
        ].values,

    "월간구매표시_하한":
        df[
            "월간구매표시_하한_분석"
        ].values,

    "BSR_대분류":
        df["BSR_분석"].values,

    "BSR_대분류명":
        df[
            "BSR_대분류명_분석"
        ].values,

    "BSR_하위순위":
        df[
            "BSR_하위순위_분석"
        ].values,

    "BSR_하위분류명":
        df[
            "BSR_하위분류명_분석"
        ].values,

    "LumaiScope_카테고리":
        df[
            "LumaiScope_카테고리"
        ].values,

    # --------------------------------------------
    # 핵심 결과
    # --------------------------------------------

    "BSR기반_월간판매량_추정":
        df[
            "BSR기반_월간판매량_추정"
        ].values,

    "BSR기반_월간판매량_하한":
        df[
            "BSR기반_월간판매량_하한"
        ].values,

    "BSR기반_월간판매량_상한":
        df[
            "BSR기반_월간판매량_상한"
        ].values,

    "BSR기반_월간매출_USD":
        df[
            "BSR기반_월간매출_USD"
        ].values,

    "BSR기반_월간매출_하한_USD":
        df[
            "BSR기반_월간매출_하한_USD"
        ].values,

    "BSR기반_월간매출_상한_USD":
        df[
            "BSR기반_월간매출_상한_USD"
        ].values,

    # --------------------------------------------
    # API 모델 정보
    # --------------------------------------------

    "BSR추정_일판매량":
        df[
            "BSR추정_일판매량"
        ].values,

    "BSR추정_모델":
        df[
            "BSR추정_모델"
        ].values,

    "BSR추정_표본수":
        df[
            "BSR추정_표본수"
        ].values,

    "BSR추정_R2":
        df[
            "BSR추정_R2"
        ].values,

    "BSR추정_기준일":
        df[
            "BSR추정_기준일"
        ].values,

    "BSR추정_외삽여부":
        df[
            "BSR추정_외삽여부"
        ].values,

    "BSR모델_상태":
        df[
            "BSR모델_상태"
        ].values,

})


# ============================================================
# 24. 경쟁상품 비교
# ============================================================

경쟁상품비교 = (
    상품통합
    .sort_values(
        by=[
            "BSR기반_월간판매량_추정",
            "BSR"
            if "BSR" in 상품통합.columns
            else "BSR_대분류"
        ],
        ascending=[
            False,
            True
        ],
        na_position="last"
    )
    .reset_index(drop=True)
)


경쟁상품비교.insert(
    0,
    "월간판매량_추정순위",
    range(
        1,
        len(경쟁상품비교) + 1
    )
)


# ============================================================
# 25. BSR 분석
# ============================================================

BSR분석 = (
    상품통합
    .sort_values(
        by="BSR_대분류",
        ascending=True,
        na_position="last"
    )
    .reset_index(drop=True)
)


# ============================================================
# 26. 카테고리별 요약
# ============================================================

BSR카테고리요약 = (
    상품통합
    .groupby(
        [
            "BSR_대분류명",
            "LumaiScope_카테고리"
        ],
        dropna=False
    )
    .agg(

        상품수=(
            "ASIN",
            "nunique"
        ),

        평균_BSR=(
            "BSR_대분류",
            "mean"
        ),

        중앙_BSR=(
            "BSR_대분류",
            "median"
        ),

        월간판매량_추정합계=(
            "BSR기반_월간판매량_추정",
            "sum"
        ),

        월간매출_추정합계_USD=(
            "BSR기반_월간매출_USD",
            "sum"
        ),

        월간매출_하한_USD=(
            "BSR기반_월간매출_하한_USD",
            "sum"
        ),

        월간매출_상한_USD=(
            "BSR기반_월간매출_상한_USD",
            "sum"
        ),

        평균_R2=(
            "BSR추정_R2",
            "mean"
        ),

        평균_모델표본수=(
            "BSR추정_표본수",
            "mean"
        )

    )
    .reset_index()
)


# ============================================================
# 27. 카테고리 시장분석
# ============================================================

카테고리시장분석 = pd.DataFrame({

    "지표": [

        "전체 상품 수",

        "BSR 존재 상품 수",

        "LumaiScope API 성공 상품 수",

        "API 성공률",

        "BSR 기반 추정 월간 판매량",

        "BSR 기반 추정 월간 매출 USD",

        "BSR 기반 월간 매출 하한 USD",

        "BSR 기반 월간 매출 상한 USD",

    ],

    "값": [

        len(df),

        df[
            "BSR_분석"
        ].notna().sum(),

        success_mask.sum(),

        (
            success_mask.sum()
            / len(df)
            if len(df)
            else np.nan
        ),

        total_sales,

        total_revenue,

        total_revenue_low,

        total_revenue_high,

    ]

})


# ============================================================
# 28. API 결과 상세
# ============================================================

BSR_API_결과 = 상품통합[
    [
        "ASIN",
        "BSR_대분류",
        "BSR_대분류명",
        "LumaiScope_카테고리",
        "BSR기반_월간판매량_추정",
        "BSR기반_월간판매량_하한",
        "BSR기반_월간판매량_상한",
        "BSR추정_일판매량",
        "BSR추정_모델",
        "BSR추정_표본수",
        "BSR추정_R2",
        "BSR추정_기준일",
        "BSR추정_외삽여부",
        "BSR모델_상태",
    ]
].copy()


# ============================================================
# 29. 추정방법
# ============================================================

추정방법 = pd.DataFrame({

    "항목": [

        "판매량 추정 방식",

        "API",

        "Marketplace",

        "BSR",

        "Category",

        "월간 판매량",

        "Likely range",

        "월간 매출",

        "주의사항",

    ],

    "내용": [

        "LumaiScope 공개 BSR Sales Estimator API 사용",

        API_URL,

        "Amazon.com (US)",

        "Amazon listing의 Top-level/main category BSR",

        "LumaiScope API가 지원하는 top-level category로 매핑",

        "API의 monthly 값",

        "API의 low ~ high 값",

        "monthly sales × 판매가격(USD)",

        "Amazon 공식 판매량이 아닌 외부 BSR 기반 추정치",

    ]

})


# ============================================================
# 30. API 실패 목록
# ============================================================

API실패 = df[
    df["BSR_API_상태"] != "SUCCESS"
][
    [
        "ASIN_분석",
        "BSR_분석",
        "BSR_대분류명_분석",
        "LumaiScope_카테고리",
        "BSR_API_상태",
        "BSR모델_상태"
    ]
].copy()


# ============================================================
# 31. Excel 저장
# ============================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = (
    f"Amazon_시장분석_LumaiScope_BSR_{timestamp}.xlsx"
)


print("\n" + "=" * 80)
print("💾 Excel 저장 중...")
print("=" * 80)


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    상품통합.to_excel(
        writer,
        sheet_name="상품통합",
        index=False
    )

    경쟁상품비교.to_excel(
        writer,
        sheet_name="경쟁상품비교",
        index=False
    )

    카테고리시장분석.to_excel(
        writer,
        sheet_name="카테고리시장분석",
        index=False
    )

    BSR분석.to_excel(
        writer,
        sheet_name="BSR분석",
        index=False
    )

    BSR카테고리요약.to_excel(
        writer,
        sheet_name="BSR카테고리요약",
        index=False
    )

    BSR_API_결과.to_excel(
        writer,
        sheet_name="BSR_API_결과",
        index=False
    )

    API실패.to_excel(
        writer,
        sheet_name="BSR_API_실패",
        index=False
    )

    추정방법.to_excel(
        writer,
        sheet_name="추정방법",
        index=False
    )

    # API 결과가 들어간 전체 데이터
    df.to_excel(
        writer,
        sheet_name="원본데이터_API결과",
        index=False
    )


# ============================================================
# 32. TOP 20 출력
# ============================================================

print("\n" + "=" * 80)
print("🏆 LumaiScope BSR 월간판매량 추정 TOP 20")
print("=" * 80)


top20 = (
    상품통합[
        [
            "ASIN",
            "브랜드",
            "상품명",
            "BSR_대분류",
            "BSR_대분류명",
            "LumaiScope_카테고리",
            "BSR기반_월간판매량_추정",
            "BSR기반_월간판매량_하한",
            "BSR기반_월간판매량_상한",
            "판매가격_USD",
            "BSR기반_월간매출_USD",
            "BSR추정_모델",
            "BSR추정_R2"
        ]
    ]
    .sort_values(
        "BSR기반_월간판매량_추정",
        ascending=False,
        na_position="last"
    )
    .head(20)
)


print(
    top20.to_string(
        index=False
    )
)


# ============================================================
# 33. 다운로드
# ============================================================

print("\n" + "=" * 80)
print("📥 다운로드")
print("=" * 80)

print(
    f"파일명: {output_file}"
)

files.download(
    output_file
)